In [1]:
import re
from collections import Counter

class BPETokenizer:

    def __init__(self, num_merges=10):
        self.num_merges = num_merges
        self.merge_rules = []
        self.vocab = {}

    # Step 1-5: Build Vocabulary
    def build_vocab(self, corpus):

        words = []

        for line in corpus:
            line = line.strip().lower()

            if line == "":
                continue

            line = re.sub(r'[^a-z]', '', line)

            if line:
                words.append(line)

        word_freq = Counter(words)

        vocab = {}

        for word, freq in word_freq.items():
            tokenized = " ".join(list(word)) + " </w>"
            vocab[tokenized] = freq

        return vocab

    # Step 6-7
    def get_pair_counts(self, vocab):

        pairs = Counter()

        for word, freq in vocab.items():

            symbols = word.split()

            for i in range(len(symbols)-1):
                pair = (symbols[i], symbols[i+1])
                pairs[pair] += freq

        return pairs

    # Step 8-10
    def merge_vocab(self, pair, vocab):

        new_vocab = {}

        pattern = re.escape(" ".join(pair))
        replacement = "".join(pair)

        for word in vocab:

            new_word = re.sub(pattern, replacement, word)

            new_vocab[new_word] = vocab[word]

        return new_vocab

    # Step 11-12
    def train(self, corpus):

        vocab = self.build_vocab(corpus)

        print("="*60)
        print("Initial Vocabulary")
        print("="*60)

        for word, freq in vocab.items():
            print(word, ":", freq)

        for i in range(self.num_merges):

            pairs = self.get_pair_counts(vocab)

            if len(pairs) == 0:
                break

            best_pair = max(pairs, key=pairs.get)

            print(f"\nMerge {i+1}: {best_pair}")

            self.merge_rules.append(best_pair)

            vocab = self.merge_vocab(best_pair, vocab)

        self.vocab = vocab

    # Step 13
    def final_vocab(self):

        vocabulary = set()

        for word in self.vocab:

            for token in word.split():
                vocabulary.add(token)

        return sorted(vocabulary)

    # Step 14
    def encode(self, word):

        word = word.lower()

        tokens = list(word)
        tokens.append("</w>")

        for pair in self.merge_rules:

            i = 0

            while i < len(tokens)-1:

                if tokens[i] == pair[0] and tokens[i+1] == pair[1]:

                    tokens[i:i+2] = ["".join(pair)]

                else:
                    i += 1

        return tokens

    # Step 15
    def decode(self, tokens):

        word = ""

        for token in tokens:

            if token != "</w>":
                word += token

        return word

    # Step 17
    def display(self):

        print("\n")
        print("="*60)
        print("Merge Rules")
        print("="*60)

        for i, rule in enumerate(self.merge_rules,1):
            print(f"{i}. {rule}")

        print("\n")
        print("="*60)
        print("Final Vocabulary")
        print("="*60)

        vocab = self.final_vocab()

        for token in vocab:
            print(token)

        print("\nVocabulary Size :", len(vocab))


# -----------------------------
# MAIN PROGRAM
# -----------------------------

corpus = [
    "low",
    "lower",
    "lowest",
    "new",
    "newer",
    "widest"
]

tokenizer = BPETokenizer(num_merges=10)

tokenizer.train(corpus)

tokenizer.display()

print("\n")
print("="*60)
print("Encoding and Decoding")
print("="*60)

test_words = [
    "lower",
    "lowest",
    "newer",
    "widest"
]

for word in test_words:

    encoded = tokenizer.encode(word)

    decoded = tokenizer.decode(encoded)

    print("\nOriginal :", word)
    print("Encoded  :", encoded)
    print("Decoded  :", decoded)

Initial Vocabulary
l o w </w> : 1
l o w e r </w> : 1
l o w e s t </w> : 1
n e w </w> : 1
n e w e r </w> : 1
w i d e s t </w> : 1

Merge 1: ('l', 'o')

Merge 2: ('lo', 'w')

Merge 3: ('low', 'e')

Merge 4: ('r', '</w>')

Merge 5: ('s', 't')

Merge 6: ('st', '</w>')

Merge 7: ('n', 'e')

Merge 8: ('ne', 'w')

Merge 9: ('low', '</w>')

Merge 10: ('lowe', 'r</w>')


Merge Rules
1. ('l', 'o')
2. ('lo', 'w')
3. ('low', 'e')
4. ('r', '</w>')
5. ('s', 't')
6. ('st', '</w>')
7. ('n', 'e')
8. ('ne', 'w')
9. ('low', '</w>')
10. ('lowe', 'r</w>')


Final Vocabulary
</w>
d
e
i
low</w>
lowe
lower</w>
new
r</w>
st</w>
w

Vocabulary Size : 11


Encoding and Decoding

Original : lower
Encoded  : ['lower</w>']
Decoded  : lower</w>

Original : lowest
Encoded  : ['lowe', 'st</w>']
Decoded  : lowest</w>

Original : newer
Encoded  : ['new', 'e', 'r</w>']
Decoded  : newer</w>

Original : widest
Encoded  : ['w', 'i', 'd', 'e', 'st</w>']
Decoded  : widest</w>
